In [0]:
%sql
USE CATALOG workspace;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS project_22b;

In [0]:
%sql
USE SCHEMA project_22b;

In [0]:
%sql
SELECT CURRENT_CATALOG(),CURRENT_SCHEMA();

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS project_22b_data;

In [0]:
%sql
SELECT * 
FROM read_files(
    '/Volumes/workspace/project_22b/project_22b_data/raw_late_facts.json',
    FORMAT => 'json'
);

In [0]:
%sql
CREATE TABLE STG_LATE_FACTS
(
    TXN_ID STRING,
    CUSTOMER_ID STRING,
    AMOUNT DECIMAL(10,2),
    TXN_DATE DATE
);

In [0]:
%sql
INSERT INTO STG_LATE_FACTS(TXN_ID,
                           CUSTOMER_ID,
                           AMOUNT,
                           TXN_DATE)
SELECT  TXN_ID,
        CUSTOMER_ID,
        AMOUNT,
        TXN_DATE
FROM read_files(
    '/Volumes/workspace/project_22b/project_22b_data/raw_late_facts.json',
    FORMAT => 'json'
);

In [0]:
%sql
SELECT  'STG_LATE_FACTS' AS STAGING_TABLE,
        COUNT(*) AS RECORD_COUNT
FROM STG_LATE_FACTS;

In [0]:
%sql
CREATE TABLE DIM_CUSTOMER_SCD2
(
    CUSTOMER_SK STRING,
    CUSTOMER_ID STRING,
    CUSTOMER_NAME STRING,
    SEGMENT STRING,
    IS_INFERRED_MEMBER BOOLEAN
);

In [0]:
%sql
INSERT INTO DIM_CUSTOMER_SCD2(CUSTOMER_SK,
                             CUSTOMER_ID,
                             CUSTOMER_NAME,
                             SEGMENT,
                             IS_INFERRED_MEMBER)
SELECT  CONCAT('CUST_SK_',(100+ROW_NUMBER() OVER(ORDER BY CUSTOMER_ID DESC))) AS CUSTOMER_SK,
        CUSTOMER_ID,
        'UNKNOWN INFERRED' AS CUSTOMER_NAME,
        'UNKNOWN' AS SEGMENT,
        TRUE AS IS_INFERRED_MEMBER
FROM (SELECT DISTINCT CUSTOMER_ID 
      FROM STG_LATE_FACTS);

In [0]:
%sql
SELECT * 
FROM dim_customer_scd2;

In [0]:
%sql
CREATE TABLE FACT_TRANSACTIONS
(
    TXN_ID STRING,
    CUSTOMER_SK STRING,
    AMOUNT DECIMAL(10,2),
    TXN_DATE DATE
);

In [0]:
%sql
INSERT INTO FACT_TRANSACTIONS(TXN_ID,
                              CUSTOMER_SK,
                              AMOUNT,
                              TXN_DATE)
SELECT  STG.TXN_ID,
        DIM.CUSTOMER_SK,
        STG.AMOUNT,
        STG.TXN_DATE
FROM STG_LATE_FACTS STG
JOIN DIM_CUSTOMER_SCD2 DIM
ON STG.CUSTOMER_ID=DIM.CUSTOMER_ID;

In [0]:
%sql
SELECT *
FROM FACT_TRANSACTIONS
ORDER BY TXN_ID;

In [0]:
%sql
MERGE INTO DIM_CUSTOMER_SCD2 TGT
USING (SELECT CUSTOMER_ID,CUSTOMER_NAME,SEGMENT,EFFECTIVE_DATE
       FROM read_files(
        '/Volumes/workspace/project_22b/project_22b_data/late_arriving_dim.json',
        FORMAT => 'json'
       )) SRC
ON TGT.CUSTOMER_ID = SRC.CUSTOMER_ID
WHEN MATCHED AND TGT.IS_INFERRED_MEMBER = TRUE THEN
    UPDATE 
        SET TGT.CUSTOMER_NAME = SRC.CUSTOMER_NAME,
            TGT.SEGMENT = SRC.SEGMENT,
            TGT.IS_INFERRED_MEMBER = FALSE;

In [0]:
%sql
SELECT * FROM DIM_CUSTOMER_SCD2
ORDER BY CUSTOMER_SK;

In [0]:
%sql
SELECT  F.TXN_ID,
        F.CUSTOMER_SK,
        D.CUSTOMER_NAME,
        D.SEGMENT,
        F.AMOUNT
FROM FACT_TRANSACTIONS F
JOIN DIM_CUSTOMER_SCD2 D
ON F.CUSTOMER_SK = D.CUSTOMER_SK;

In [0]:
%sql
SELECT  'NULL_CUSTOMER_SK' AS RULE_NAME,
        'REFERENTIAL' AS CHECK_TYPE,
        CASE WHEN COUNT(*)=0
        THEN 'PASSED'
        ELSE 'FAILED'
        END AS STATUS
FROM FACT_TRANSACTIONS
WHERE CUSTOMER_SK IS NULL
UNION ALL
SELECT  'VALID_AMOUNT',
        'RANGE',
        CASE WHEN COUNT(*)=0
        THEN 'PASSED' 
        ELSE 'FAILED'
        END AS STATUS
FROM fact_transactions
WHERE AMOUNT<=0 OR AMOUNT IS NULL
UNION ALL
SELECT  'UNRESOLVED _INFER',
        'INTEGRITY',
        CASE WHEN COUNT(*)=0
        THEN 'PASSED'
        ELSE 'FAILED'
        END AS STATUS
FROM DIM_CUSTOMER_SCD2
WHERE IS_INFERRED_MEMBER = TRUE;

In [0]:
%sql
DESCRIBE HISTORY DIM_CUSTOMER_SCD2;

In [0]:
%sql
SELECT  D1.CUSTOMER_ID,
        D1.CUSTOMER_NAME AS PREV_STATE,
        CONCAT(D2.CUSTOMER_NAME,' (',LEFT(D2.SEGMENT,3),')') AS CURRENT_STATE
FROM DIM_CUSTOMER_SCD2 VERSION AS OF 1 D1
JOIN DIM_CUSTOMER_SCD2 VERSION AS OF 2 D2
ON D1.CUSTOMER_SK = D2.CUSTOMER_SK;


In [0]:
%sql
OPTIMIZE DIM_CUSTOMER_SCD2
ZORDER BY (CUSTOMER_SK);

In [0]:
%sql
DESCRIBE HISTORY DIM_CUSTOMER_SCD2;

In [0]:
%sql
SELECT  'DIM_CUSTOMER_SCD2' AS TARGET_TABLE,
        'OPTIMIZED' AS CLUSTERING_STATUS;